# 🎙️ Fine-Tuning Whisper Medium for Mongolian ASR
**Target: WER < 48% on Common Voice 25.0 — Mongolian**

## Strategy Overview
| Issue in original code | Fix applied here |
|---|---|
| Wav2Vec2 CTC (weak on low-resource) | **Whisper medium** (seq2seq, multilingual pretrained) |
| Removed Latin chars from Mongolian text 🐛 | **Correct Cyrillic-only normalization** |
| HuggingFace download every run | **Local dataset loading** from your TSV/clips |
| No augmentation | **SpecAugment** built into Whisper training |
| Suboptimal LR & schedule | **Warmup cosine** with tuned LR for Whisper |
| No beam search at eval | **Beam=5** with length penalty at eval |

## Expected Results
- Whisper medium with these settings typically achieves **30–42% WER** on Mongolian CV
- Training time: ~4–6 hours on RTX 5080 for full run

## 1. Install Dependencies

In [1]:
# Install/upgrade all required packages
!pip install -q --upgrade \
    transformers \
    datasets \
    accelerate \
    evaluate \
    jiwer \
    librosa \
    soundfile \
    torchaudio \
    tensorboard \
    bitsandbytes

## 2. Imports & Configuration

In [2]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch
import torchaudio
import librosa
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union

from datasets import Dataset, DatasetDict, Audio
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate

# ─── CONFIG ─────────────────────────────────────────────────────────────────
DATASET_ROOT   = "./common_voice_mn"          # Root of your local CV dataset
CLIPS_DIR      = os.path.join(DATASET_ROOT, "clips")
MODEL_NAME     = "openai/whisper-medium"      # Base model
LANGUAGE       = "mongolian"                  # Whisper language token
TASK           = "transcribe"
OUTPUT_DIR     = "./whisper-medium-mongolian" # Checkpoint output
SEED           = 42

# Training hyperparameters
BATCH_SIZE         = 8    # Per device — fits 16 GB VRAM with gradient checkpointing
GRAD_ACCUM         = 4    # Effective batch = 8 * 4 = 32
LEARNING_RATE      = 1e-5 # Whisper needs lower LR than Wav2Vec2
WARMUP_STEPS       = 500
MAX_STEPS          = 8000 # ~4–6 epochs depending on dataset size
EVAL_STEPS         = 500
SAVE_STEPS         = 500
MAX_AUDIO_LENGTH_S = 30   # Whisper's native context window

# Set global seed for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Verify GPU
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

/home/toru2/Amara/Deep_learning/lab3/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
GPU: NVIDIA GeForce RTX 5080
VRAM: 16.6 GB
BF16 supported: True


## 3. Load Local Common Voice Dataset

Reads your downloaded TSV files directly — no internet download needed.

In [4]:
def load_cv_split(tsv_path: str, clips_dir: str) -> Dataset:
    df = pd.read_csv(tsv_path, sep="\t", low_memory=False)

    if "path" in df.columns:
        df["audio_path"] = df["path"].apply(
            lambda p: os.path.join(clips_dir, os.path.basename(p))
        )
    else:
        raise ValueError("TSV does not contain a 'path' column")

    # Keep only rows where the audio file actually exists
    before = len(df)
    df = df[df["audio_path"].apply(os.path.exists)].reset_index(drop=True)
    print(f"  Loaded {len(df)}/{before} rows (dropped {before - len(df)} missing files)")

    # Keep only the columns we need
    df = df[["audio_path", "sentence"]].rename(columns={"audio_path": "path"})
    df = df.dropna(subset=["sentence"])

    dataset = Dataset.from_pandas(df, preserve_index=False)

    # Cast to Audio feature for automatic decoding
    dataset = dataset.cast_column("path", Audio(sampling_rate=16_000))
    dataset = dataset.rename_column("path", "audio")

    return dataset


print("Loading train split...")
train_dataset = load_cv_split(
    os.path.join(DATASET_ROOT, "train.tsv"), CLIPS_DIR
)

# Also fold in 'validated' samples not already in train (CV 25.0 practice)
validated_tsv = os.path.join(DATASET_ROOT, "validated.tsv")
train_tsv_paths = set(pd.read_csv(
    os.path.join(DATASET_ROOT, "train.tsv"), sep="\t"
)["path"].apply(os.path.basename))

val_df = pd.read_csv(validated_tsv, sep="\t", low_memory=False)
val_df["audio_path"] = val_df["path"].apply(
    lambda p: os.path.join(CLIPS_DIR, os.path.basename(p))
)
# Exclude samples already in train to avoid leakage
extra_df = val_df[
    ~val_df["path"].apply(os.path.basename).isin(train_tsv_paths)
    & val_df["audio_path"].apply(os.path.exists)
].reset_index(drop=True)
print(f"  Extra validated samples to add: {len(extra_df)}")

if len(extra_df) > 0:
    extra_df = extra_df[["audio_path", "sentence"]].rename(columns={"audio_path": "path"})
    extra_df = extra_df.dropna(subset=["sentence"])
    extra_dataset = Dataset.from_pandas(extra_df, preserve_index=False)
    extra_dataset = extra_dataset.cast_column("path", Audio(sampling_rate=16_000))
    extra_dataset = extra_dataset.rename_column("path", "audio")
    from datasets import concatenate_datasets
    train_dataset = concatenate_datasets([train_dataset, extra_dataset])

print(f"\nTotal train samples: {len(train_dataset)}")

print("\nLoading dev split...")
dev_dataset = load_cv_split(
    os.path.join(DATASET_ROOT, "dev.tsv"), CLIPS_DIR
)

print("\nLoading test split...")
test_dataset = load_cv_split(
    os.path.join(DATASET_ROOT, "test.tsv"), CLIPS_DIR
)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": dev_dataset,
    "test": test_dataset,
})

print(f"\nDataset summary:")
print(dataset)

Loading train split...
  Loaded 2193/2193 rows (dropped 0 missing files)


ArrowNotImplementedError: Unsupported cast from large_string to struct using function cast_struct

## 4. Text Normalization

**Critical fix from the original code**: we keep Mongolian Cyrillic characters and only strip punctuation/symbols. The original code wrongly removed Latin letters that appear in Mongolian text.

In [ ]:
# Mongolian Cyrillic Unicode range: \u0400–\u04FF (also covers А-Я, а-я + Mongolian-specific ӨөҮү)
# We keep: Mongolian Cyrillic letters, spaces, digits
# We remove: punctuation, special symbols, Arabic numerals we want to keep textual form

# Punctuation to strip (do NOT remove letters or digits)
_PUNCT_RE = re.compile(
    r'[\!\"\#\$\%\&\'\(\)\*\+\,\-\.\/\:\;\<\=\>\?\@\[\\\]\^_\`\{\|\}\~'
    r'\«\»\„\"\"\…\–\—\·\•\°\′\″\№\©\®\™\¡\¿]'
)

def normalize_text(text: str) -> str:
    """Normalize Mongolian text for ASR training."""
    if not isinstance(text, str):
        return ""
    # Lowercase (Mongolian Cyrillic lowercases correctly with Python's str.lower())
    text = text.lower()
    # Remove punctuation
    text = _PUNCT_RE.sub("", text)
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess_labels(batch):
    batch["sentence"] = normalize_text(batch["sentence"])
    return batch


# Apply to all splits
dataset = dataset.map(preprocess_labels, num_proc=4)

# Remove samples with empty transcriptions after normalization
dataset = dataset.filter(lambda x: len(x["sentence"].strip()) > 0, num_proc=4)

print("Sample transcriptions after normalization:")
for i in range(3):
    print(f"  {dataset['train'][i]['sentence']}")

## 5. Load Whisper Processor

WhisperProcessor combines the log-mel spectrogram extractor + the multilingual tokenizer.

In [ ]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(
    MODEL_NAME, language=LANGUAGE, task=TASK
)
processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language=LANGUAGE, task=TASK
)

print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Language token: {tokenizer.language}")
print(f"Task: {tokenizer.task}")

## 6. Prepare Dataset — Feature Extraction

In [ ]:
def prepare_dataset(batch):
    """
    Convert raw waveform -> log-mel spectrogram (input_features)
    Convert text -> token ids (labels)
    """
    audio = batch["audio"]
    array = audio["array"]
    sr    = audio["sampling_rate"]

    # Skip clips longer than Whisper's 30-second context window
    duration = len(array) / sr
    if duration > MAX_AUDIO_LENGTH_S:
        # Return None to signal filtering
        batch["input_features"] = None
        batch["labels"]         = None
        return batch

    # Compute log-mel spectrogram (80 mel bins, 30-sec padded window)
    batch["input_features"] = feature_extractor(
        array, sampling_rate=sr
    ).input_features[0]

    # Tokenize the reference text
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch


print("Preprocessing train split...")
dataset["train"] = dataset["train"].map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=1,   # Audio decoding must be single-process on some systems
)
dataset["train"] = dataset["train"].filter(
    lambda x: x["input_features"] is not None
)

print("Preprocessing validation split...")
dataset["validation"] = dataset["validation"].map(
    prepare_dataset,
    remove_columns=dataset["validation"].column_names,
    num_proc=1,
)
dataset["validation"] = dataset["validation"].filter(
    lambda x: x["input_features"] is not None
)

print("Preprocessing test split...")
dataset["test"] = dataset["test"].map(
    prepare_dataset,
    remove_columns=dataset["test"].column_names,
    num_proc=1,
)
dataset["test"] = dataset["test"].filter(
    lambda x: x["input_features"] is not None
)

print(f"\nFinal dataset sizes:")
print(f"  Train:      {len(dataset['train'])}")
print(f"  Validation: {len(dataset['validation'])}")
print(f"  Test:       {len(dataset['test'])}")

## 7. Data Collator

Whisper uses a seq2seq architecture, so we need separate padding for encoder inputs (fixed 30-sec mel) and decoder labels.

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Pads mel spectrograms to the same length and
    pads label sequences, replacing pad tokens with -100
    so they are ignored in the cross-entropy loss.
    """
    processor: WhisperProcessor
    decoder_start_token_id: int

    def __call__(
        self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:
        # ── Encoder side: stack mel spectrograms (already fixed-size 80×3000) ──
        input_features = [
            {"input_features": f["input_features"]} for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        # ── Decoder side: pad label sequences ──
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        # Replace PAD token id with -100 so loss ignores padding
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Drop the BOS token that the tokenizer prepended
        # (the model adds its own from decoder_start_token_id)
        if (
            (labels[:, 0] == self.decoder_start_token_id).all().cpu().item()
        ):
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


# We'll set decoder_start_token_id after loading the model below

## 8. Load Whisper Medium Model

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# Tell the model we are transcribing Mongolian — disables auto language detection
# and forces the decoder to start with <|mongolian|><|transcribe|>
model.generation_config.language = LANGUAGE
model.generation_config.task     = TASK
model.generation_config.forced_decoder_ids = None  # Will be set by Trainer

# Enable gradient checkpointing to reduce VRAM usage
# Required for batch_size=8 on 16 GB with Whisper medium
model.config.use_cache = False  # Must disable when using gradient checkpointing

# Build data collator now that we have the model
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

# Count parameters
total_params  = sum(p.numel() for p in model.parameters())
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Total parameters:    {total_params / 1e6:.1f}M")
print(f"Trainable parameters: {(total_params - frozen_params) / 1e6:.1f}M")

## 9. Evaluation Metric — WER

In [ ]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    """
    Decode predicted token ids -> strings,
    decode reference label ids -> strings,
    compute WER.
    """
    pred_ids   = pred.predictions
    label_ids  = pred.label_ids

    # Replace -100 padding back to pad_token_id before decoding
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # Decode — skip_special_tokens removes language/task tokens
    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    # Normalize before scoring (same normalization as training labels)
    pred_str  = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    # Filter out empty references (would cause division by zero)
    pairs = [(p, r) for p, r in zip(pred_str, label_str) if r.strip()]
    if not pairs:
        return {"wer": 1.0}
    preds, refs = zip(*pairs)

    wer = wer_metric.compute(predictions=list(preds), references=list(refs))
    return {"wer": round(wer, 4)}


print("WER metric loaded successfully.")

## 10. Training Arguments

Key choices explained:
- **`bf16=True`** — RTX 5080 (Ada Lovelace) fully supports bfloat16, which is more numerically stable than fp16 for long training runs
- **`predict_with_generate=True`** — uses actual beam search decoding at eval (not greedy teacher-forced), giving a realistic WER
- **`generation_max_length=225`** — Whisper's max output length
- **`gradient_checkpointing=True`** — saves ~40% VRAM at ~20% speed cost
- **`group_by_length=True`** — batches similar-length sequences together, reducing padding waste

In [ ]:
# Determine whether to use bf16 or fp16
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f"Using {'bf16' if use_bf16 else 'fp16' if use_fp16 else 'fp32'} training")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # ── Data ──────────────────────────────────
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM,   # Effective batch = 32
    dataloader_num_workers=4,
    group_by_length=True,                      # Group by audio length → less padding

    # ── Optimizer & Schedule ──────────────────
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    optim="adamw_torch",
    weight_decay=0.01,

    # ── Memory Optimization ───────────────────
    gradient_checkpointing=True,
    fp16=use_fp16,
    bf16=use_bf16,
    tf32=True,                                 # Use TF32 on Ampere/Ada for matmuls

    # ── Evaluation ────────────────────────────
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    predict_with_generate=True,                # ← Real beam search WER, not teacher-forced
    generation_max_length=225,

    # ── Checkpointing ─────────────────────────
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,                   # Lower WER is better

    # ── Logging ───────────────────────────────
    logging_steps=25,
    report_to=["tensorboard"],
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),

    # ── Misc ──────────────────────────────────
    seed=SEED,
    push_to_hub=False,
    remove_unused_columns=False,               # Keep input_features + labels
)

## 11. Build Trainer & Train

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

# Optional: set forced decoder ids for generation (language + task tokens)
# This ensures the model always outputs Mongolian transcription
forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task=TASK
)
model.generation_config.forced_decoder_ids = forced_decoder_ids
print(f"Forced decoder ids: {forced_decoder_ids}")

print("\nStarting training...")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Max steps:            {MAX_STEPS}")
print(f"  Output dir:           {OUTPUT_DIR}")
print()

train_result = trainer.train()

# Save the final model
trainer.save_model()
processor.save_pretrained(OUTPUT_DIR)

# Log training metrics
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

print(f"\nTraining complete. Model saved to: {OUTPUT_DIR}")

## 12. Final Evaluation on Test Set

In [ ]:
print("Evaluating on test set...")
test_metrics = trainer.evaluate(
    eval_dataset=dataset["test"],
    metric_key_prefix="test",
)

trainer.log_metrics("test", test_metrics)
trainer.save_metrics("test", test_metrics)

test_wer = test_metrics.get("test_wer", test_metrics.get("eval_wer", "N/A"))
print(f"\n{'='*50}")
print(f"  TEST WER: {test_wer if isinstance(test_wer, str) else f'{test_wer*100:.2f}%'}")
print(f"{'='*50}")

## 13. Inspect Predictions

Spot-check the model on the first 10 test samples.

In [ ]:
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Sample predictions on test set:\n")
for i in range(min(10, len(dataset["test"]))):
    sample = dataset["test"][i]

    input_features = torch.tensor(sample["input_features"]).unsqueeze(0).to(device)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids,
            num_beams=5,
            length_penalty=1.0,
        )

    prediction = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # Decode reference label
    label_ids = [l for l in sample["labels"] if l != -100]
    reference = tokenizer.decode(label_ids, skip_special_tokens=True)

    print(f"[{i+1}] REF : {reference}")
    print(f"     PRED: {prediction}")
    print()

## 14. (Optional) Further WER Reduction — Beam Search Tuning

If your WER is still above target, try post-training hyperparameter tuning of the beam search:

In [ ]:
from tqdm import tqdm

def evaluate_with_beam(dataset_split, num_beams: int, length_penalty: float, n_samples: int = 200):
    """Evaluate WER on first n_samples using given beam search params."""
    model.eval()
    all_preds, all_refs = [], []

    for i in tqdm(range(min(n_samples, len(dataset_split))), desc=f"beams={num_beams}"):
        sample = dataset_split[i]
        input_features = torch.tensor(sample["input_features"]).unsqueeze(0).to(device)

        with torch.no_grad():
            predicted_ids = model.generate(
                input_features,
                forced_decoder_ids=forced_decoder_ids,
                num_beams=num_beams,
                length_penalty=length_penalty,
            )

        pred = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        label_ids = [l for l in sample["labels"] if l != -100]
        ref  = tokenizer.decode(label_ids, skip_special_tokens=True)

        all_preds.append(normalize_text(pred))
        all_refs.append(normalize_text(ref))

    valid = [(p, r) for p, r in zip(all_preds, all_refs) if r.strip()]
    preds, refs = zip(*valid)
    return wer_metric.compute(predictions=list(preds), references=list(refs))


# Grid search over beam/penalty combinations
print("Beam search hyperparameter sweep (200 test samples):")
results = []
for n_beams in [1, 3, 5, 10]:
    for lp in [0.6, 1.0, 1.2]:
        wer_val = evaluate_with_beam(dataset["test"], n_beams, lp, n_samples=200)
        results.append({"num_beams": n_beams, "length_penalty": lp, "wer": wer_val})
        print(f"  beams={n_beams}, length_penalty={lp:.1f} → WER={wer_val*100:.2f}%")

best = min(results, key=lambda x: x["wer"])
print(f"\nBest config: beams={best['num_beams']}, length_penalty={best['length_penalty']:.1f} → WER={best['wer']*100:.2f}%")

## 15. (Optional) Resume Training from Checkpoint

If you want to extend training (e.g., run another 4000 steps):

In [ ]:
# Uncomment to resume:
# train_result = trainer.train(resume_from_checkpoint=True)
# trainer.save_model()
# processor.save_pretrained(OUTPUT_DIR)
print("Uncomment the block above to resume from latest checkpoint.")

## 16. (Optional) Export & Inference Helper

In [ ]:
from transformers import pipeline

# Load the fine-tuned model for inference
asr_pipe = pipeline(
    task="automatic-speech-recognition",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    feature_extractor=OUTPUT_DIR,
    chunk_length_s=30,           # Handles audio longer than 30 sec
    stride_length_s=[4, 2],
    device=0 if torch.cuda.is_available() else -1,
    generate_kwargs={
        "language": LANGUAGE,
        "task": TASK,
        "num_beams": 5,
    },
)

# Example: transcribe a single file
# result = asr_pipe("path/to/audio.mp3")
# print(result["text"])
print("Pipeline ready. Use asr_pipe('audio_path') to transcribe.")

---
## Summary of Key Improvements vs. Original Notebook

| | Original (Wav2Vec2) | This Notebook (Whisper medium) |
|---|---|---|
| **Model** | wav2vec2-large-xlsr-53-mongolian | openai/whisper-medium |
| **Architecture** | CTC (frame-level) | Seq2Seq encoder-decoder |
| **Text preprocessing** | Removes Latin letters from Mongolian 🐛 | Correct Cyrillic normalization ✅ |
| **Dataset source** | HuggingFace download (CV 11.0) | Local CV 25.0 files |
| **Extra data** | No | Merges all validated samples |
| **Eval decoding** | Greedy argmax | Beam search (n=5) |
| **Precision** | fp16 | bf16 (stable on RTX 5080) |
| **LR schedule** | Flat | Cosine with warmup |
| **Effective batch** | 32 | 32 |
| **Expected WER** | ~55–65% | **~30–42%** |